# Stage 3 — Canonicalisation

Turns raw pixel keypoints into view-invariant body kinematics. Raw coordinates encode camera geometry and where the player is standing; the model needs stroke *mechanics*.

### Transforms, in order

1. **Confidence gating** — low-confidence keypoints interpolated in time, or masked
2. **Flip** — all right-end strikers mirrored, so everyone faces the same way
3. **Hip-centring** — origin at hip midpoint, removing absolute position
4. **Torso scaling** — divide by shoulder-to-hip distance, removing camera distance
5. **Derived channels** — velocity, plus table-relative distance

### The mirroring constraint

A flip (negate x + swap L/R indices) changes **side and handedness together**, so the four combinations form two closed orbits:

```
(left end, right-handed)  <->  (right end, left-handed)
(right end, right-handed) <->  (left end, left-handed)
```

`side XOR handedness` is invariant under flipping, and it is physically real — a right-hander at the left end shows their forehand toward the camera, at the right end away. No reflection merges those.

**One binary residual is therefore unavoidable.** Side is a 50/50 split (726/730); handedness is only 4% (63 of 1457). Normalising side removes far more variance, so that is what we flip on. The residual `eff_hand` becomes an explicit feature, and `playing_wrist` tells Stage 4 which arm to measure.

### Outputs

| file | contents |
|---|---|
| `derived/clips/canonical.npz` | canonical keypoints, velocities, masks, per-stroke metadata |
| `derived/meta/canon_quality.csv` | per-video validity report |

Runs in ~1 min. No GPU needed.


## 1 · Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Config

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

CONF_MIN      = 0.35   # below this a keypoint is treated as missing
MAX_GAP       = 12     # interpolate gaps up to this many frames (0.10 s)
MIN_VALID_PCT = 0.60   # a stroke needs this fraction of usable frames

import json
from pathlib import Path
import numpy as np
import pandas as pd

BASE    = Path(BASE)
META    = BASE / "derived/meta"
POSEDIR = BASE / "derived/pose"
CLIPDIR = BASE / "derived/clips"; CLIPDIR.mkdir(parents=True, exist_ok=True)

def load(stem):
    p = META / f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META / f"{stem}.csv")

strokes = load("strokes")
players = json.loads((META / "players.json").read_text())
folds   = json.loads((META / "folds.json").read_text())
NF, PRE = folds["window"]["n_frames"], folds["window"]["pre"]

VIDEO_IDS = sorted(strokes.video_id.unique(),
                   key=lambda v: (v.split("_")[0], int(v.split("_")[1])))

# COCO-17
NOSE = 0
L_SHO, R_SHO, L_ELB, R_ELB, L_WRI, R_WRI = 5, 6, 7, 8, 9, 10
L_HIP, R_HIP, L_KNE, R_KNE, L_ANK, R_ANK = 11, 12, 13, 14, 15, 16
FLIP_PAIRS = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]

print(f"{len(strokes)} strokes | window {NF} frames, contact at index {PRE}")
print(f"left-handed players: "
      f"{[k for k,v in players.items() if v['handedness']=='L']}")

1457 strokes | window 97 frames, contact at index 60
left-handed players: ['test_3__right', 'test_5__left', 'test_5__right', 'test_7__right']


## 3 · Transforms

**Confidence gating first.** A keypoint below `CONF_MIN` is unreliable, and a wrist that jitters to a random location does more damage than a missing one. Short gaps are interpolated along the time axis; long gaps stay masked so downstream code can exclude them rather than trusting invented values.

In [3]:
def gate_and_fill(kp, sc):
    """Interpolate low-confidence keypoints over short gaps. -> kp, valid mask"""
    kp = kp.astype(np.float32).copy()
    good = sc >= CONF_MIN                       # (T, J)
    T, J = good.shape

    for j in range(J):
        g = good[:, j]
        if g.all() or not g.any():
            continue
        idx = np.arange(T)
        # only fill runs shorter than MAX_GAP
        fill = np.zeros(T, bool)
        run_start = None
        for t in range(T):
            if not g[t] and run_start is None:
                run_start = t
            elif g[t] and run_start is not None:
                if t - run_start <= MAX_GAP and run_start > 0:
                    fill[run_start:t] = True
                run_start = None
        if fill.any():
            for c in (0, 1):
                kp[fill, j, c] = np.interp(idx[fill], idx[g], kp[g, j, c])
            good[fill, j] = True
    return kp, good


def flip(kp):
    """Mirror: negate x about 0 and swap left/right joints.
    Must be applied BEFORE hip-centring is undone — here x is already
    hip-centred, so negating about 0 is the correct mirror axis."""
    out = kp.copy()
    out[..., 0] *= -1
    for a, b in FLIP_PAIRS:
        out[:, [a, b]] = out[:, [b, a]]
    return out


def canonicalise(kp, sc, do_flip):
    """-> canonical kp (T,17,2), valid (T,17), torso_px (scalar)"""
    kp, valid = gate_and_fill(kp, sc)

    hip = (kp[:, L_HIP] + kp[:, R_HIP]) / 2          # (T,2)
    sho = (kp[:, L_SHO] + kp[:, R_SHO]) / 2
    torso = np.linalg.norm(sho - hip, axis=-1)       # (T,)
    ok = torso > 1
    torso_px = float(np.median(torso[ok])) if ok.any() else np.nan
    if not np.isfinite(torso_px) or torso_px <= 0:
        return None, valid, np.nan

    kp = (kp - hip[:, None, :]) / torso_px           # centre + scale
    if do_flip:
        kp = flip(kp)
        valid = valid.copy()
        for a, b in FLIP_PAIRS:
            valid[:, [a, b]] = valid[:, [b, a]]
    return kp, valid, torso_px

print("transforms defined")

transforms defined


## 4 · Mirror-invariance test — the gate

The assertion that catches the bug which would otherwise cost ~10 points on folds F and G, silently.

Take a real stroke, synthetically mirror it, and flip its side label. Canonicalisation should map both to the **same** sequence. If it doesn't, the flip logic is wrong.

In [4]:
def _load_any():
    for vid in VIDEO_IDS:
        p = POSEDIR / f"{vid}.npz"
        if p.exists():
            d = np.load(p, allow_pickle=True)
            for i in range(len(d["stroke_id"])):
                pi = 0 if d["side"][i] == "left" else 1
                if d["detected"][i, pi]:
                    return (d["keypoints"][i, pi].astype(np.float32),
                            d["scores"][i, pi].astype(np.float32))
    raise RuntimeError("no pose cache found")

kp0, sc0 = _load_any()

# synthetic mirror of the raw input: reflect x about an arbitrary axis,
# swap L/R joints. This is the same person seen from the other end.
AXIS = 1920.0
kp_m = kp0.copy(); kp_m[..., 0] = AXIS - kp_m[..., 0]
sc_m = sc0.copy()
for a, b in FLIP_PAIRS:
    kp_m[:, [a, b]] = kp_m[:, [b, a]]
    sc_m[:, [a, b]] = sc_m[:, [b, a]]

a_, va, _ = canonicalise(kp0,  sc0,  do_flip=False)   # left-end striker
b_, vb, _ = canonicalise(kp_m, sc_m, do_flip=True)    # right-end striker, flipped
c_, vc, _ = canonicalise(kp_m, sc_m, do_flip=False)   # control: flip omitted

# --- sensitivity: without the flip the two must NOT match. If they do, the
# chosen stroke is left-right symmetric and the whole test proves nothing.
ctrl = np.abs(a_ - c_)[va & vc].max()
print(f"control (flip omitted, should be LARGE) : {ctrl:.3e}")
assert ctrl > 0.05, (
    f"TEST IS VACUOUS (control={ctrl:.4f}). The sampled stroke is nearly "
    "left-right symmetric, so it cannot detect a broken flip. Sample another.")

err = np.abs(a_ - b_)[va & vb].max()
print(f"with flip (should be ~0)               : {err:.3e}")
assert err < 1e-3, (
    f"MIRROR INVARIANCE FAILED (err={err:.4f}). The flip logic is wrong — "
    "do not proceed, every right-end stroke would be corrupted.")
print("\nPASS — the test is sensitive, and a stroke and its mirror "
      "canonicalise identically.")

control (flip omitted, should be LARGE) : 1.546e+00
with flip (should be ~0)               : 0.000e+00

PASS — the test is sensitive, and a stroke and its mirror canonicalise identically.


## 5 · Apply to all strokes

Both players are canonicalised (per decision D8), using the striker's flip so the two skeletons stay in a common frame — that keeps the option of testing whether opponent context helps defence recall.

In [5]:
N = len(strokes)
KPC  = np.zeros((N, 2, NF, 17, 2), np.float32)   # 0 = striker, 1 = opponent
VAL  = np.zeros((N, 2, NF, 17), bool)
TORSO= np.full((N, 2), np.nan, np.float32)
TDIST= np.full((N, NF), np.nan, np.float32)      # striker -> table edge
META_ROWS = []

order = {sid: i for i, sid in enumerate(strokes.stroke_id)}

for vid in VIDEO_IDS:
    p = POSEDIR / f"{vid}.npz"
    if not p.exists():
        print(f"  {vid}: no pose cache"); continue
    d = np.load(p, allow_pickle=True)
    ids, sides = list(d["stroke_id"]), d["side"]
    KP, SC, DET, BX = d["keypoints"], d["scores"], d["detected"], d["boxes"]
    table = d["table_box"]

    for i, sid in enumerate(ids):
        r = order.get(sid)
        if r is None:
            continue
        side = sides[i]
        s_pi = 0 if side == "left" else 1        # striker index
        o_pi = 1 - s_pi

        hand     = players[f"{vid}__{side}"]["handedness"]
        do_flip  = (side == "right")             # normalise side (see header)
        eff_hand = hand if not do_flip else ("R" if hand == "L" else "L")

        for slot, pi in ((0, s_pi), (1, o_pi)):
            if not DET[i, pi]:
                continue
            c, v, t = canonicalise(KP[i, pi].astype(np.float32),
                                   SC[i, pi].astype(np.float32), do_flip)
            if c is None:
                continue
            KPC[r, slot], VAL[r, slot], TORSO[r, slot] = c, v, t

        # striker distance from the near table edge, torso-normalised
        if DET[i, s_pi] and np.isfinite(TORSO[r, 0]) and table[2] > table[0]:
            hipx = (KP[i, s_pi, :, L_HIP, 0] + KP[i, s_pi, :, R_HIP, 0]) / 2
            edge = table[0] if side == "left" else table[2]
            TDIST[r] = np.abs(hipx - edge) / TORSO[r, 0]

        META_ROWS.append({
            "stroke_id": sid, "eff_hand": eff_hand,
            "playing_wrist": R_WRI if eff_hand == "R" else L_WRI,
            "flipped": bool(do_flip), "orig_hand": hand,
        })
    print(f"  {vid}: {len(ids)} strokes canonicalised")

mrows = pd.DataFrame(META_ROWS).set_index("stroke_id").reindex(strokes.stroke_id)
print(f"\ntotal {len(mrows)} rows")

  game_1: 161 strokes canonicalised
  game_2: 399 strokes canonicalised
  game_3: 153 strokes canonicalised
  game_4: 173 strokes canonicalised
  game_5: 248 strokes canonicalised
  test_1: 84 strokes canonicalised
  test_2: 29 strokes canonicalised
  test_3: 24 strokes canonicalised
  test_4: 71 strokes canonicalised
  test_5: 27 strokes canonicalised
  test_6: 39 strokes canonicalised
  test_7: 49 strokes canonicalised

total 1457 rows


## 6 · Quality report

In [6]:
valid_pct   = VAL[:, 0].mean(axis=(1, 2))
wrist_valid = VAL[np.arange(len(strokes)), 0][
    :, :, [L_WRI, R_WRI]].mean(axis=(1, 2))
usable = (valid_pct >= MIN_VALID_PCT) & np.isfinite(TORSO[:, 0])

q = strokes.copy()
q["valid_pct"], q["wrist_valid"], q["usable"] = valid_pct, wrist_valid, usable
rep = (q.groupby("video_id")
         .agg(n=("stroke_id", "size"),
              valid_pct=("valid_pct", "mean"),
              wrist_valid=("wrist_valid", "mean"),
              usable=("usable", "mean"))
         .reindex(VIDEO_IDS))
rep.to_csv(META / "canon_quality.csv")
print(rep.to_string(float_format=lambda x: f"{x:.3f}"))

print("\n" + "=" * 68)
print(f"  usable strokes : {usable.sum()} / {len(strokes)} "
      f"({100*usable.mean():.1f}%)")
print(f"  mean keypoint validity : {valid_pct.mean():.3f}")
print(f"  mean wrist validity    : {wrist_valid.mean():.3f}")
bad = q[~q.usable]
if len(bad):
    print(f"\n  dropped ({len(bad)}): {dict(bad.video_id.value_counts())}")
    print(f"  by class: {dict(bad.shot_class.value_counts())}")
print("\n  fold x class AFTER dropping unusable:")
print(pd.crosstab(q[q.usable].fold, q[q.usable].shot_class).to_string())
print("=" * 68)

            n  valid_pct  wrist_valid  usable
video_id                                     
game_1    161      0.991        0.995   0.994
game_2    399      0.960        0.962   0.965
game_3    153      0.991        0.989   0.993
game_4    173      0.975        0.971   0.983
game_5    248      1.000        0.999   1.000
test_1     84      0.996        0.997   1.000
test_2     29      1.000        1.000   1.000
test_3     24      0.993        0.991   1.000
test_4     71      0.991        0.980   1.000
test_5     27      0.888        0.883   0.889
test_6     39      0.957        0.963   0.974
test_7     49      0.958        0.957   0.959

  usable strokes : 1432 / 1457 (98.3%)
  mean keypoint validity : 0.979
  mean wrist validity    : 0.978

  dropped (25): {'game_2': np.int64(14), 'test_5': np.int64(3), 'game_4': np.int64(3), 'test_7': np.int64(2), 'game_1': np.int64(1), 'game_3': np.int64(1), 'test_6': np.int64(1)}
  by class: {'serve': np.int64(13), 'attack': np.int64(10), 'defence':

## 7 · Save

Velocity is stored alongside position because it is the single most discriminative signal for attack-vs-control, and computing it once here keeps every downstream stage consistent.

In [7]:
VEL = np.zeros_like(KPC)
VEL[:, :, 1:] = np.diff(KPC, axis=2)             # per-frame displacement

np.savez_compressed(
    CLIPDIR / "canonical.npz",
    stroke_id=strokes.stroke_id.values.astype(str),
    video_id=strokes.video_id.values.astype(str),
    fold=strokes.fold.values.astype(str),
    shot_class=strokes.shot_class.values.astype(str),
    high_level=strokes.high_level.values.astype(str),
    technique=strokes.technique.values.astype(str),
    lean=strokes.lean.astype(str).values,
    feet=strokes.feet.astype(str).values,
    kp=KPC.astype(np.float16), vel=VEL.astype(np.float16),
    valid=VAL, torso=TORSO, table_dist=TDIST.astype(np.float16),
    eff_hand=mrows.eff_hand.values.astype(str),
    playing_wrist=mrows.playing_wrist.values.astype(np.int8),
    flipped=mrows.flipped.values.astype(bool),
    usable=usable,
    contact_index=PRE, n_frames=NF,
)

out = CLIPDIR / "canonical.npz"
print(f"-> {out}  ({out.stat().st_size/1e6:.1f} MB)")
print(f"   kp    {KPC.shape}   [stroke, player(0=striker), frame, joint, xy]")
print(f"   vel   {VEL.shape}")
print(f"   valid {VAL.shape}")
print(f"\n   eff_hand: {dict(pd.Series(mrows.eff_hand).value_counts())}")
print(f"   flipped : {int(mrows.flipped.sum())} / {len(mrows)}")

-> /content/drive/MyDrive/tt_coach/derived/clips/canonical.npz  (25.6 MB)
   kp    (1457, 2, 97, 17, 2)   [stroke, player(0=striker), frame, joint, xy]
   vel   (1457, 2, 97, 17, 2)
   valid (1457, 2, 97, 17)

   eff_hand: {'R': np.int64(765), 'L': np.int64(692)}
   flipped : 731 / 1457


---
## Done

| artifact | used by |
|---|---|
| `derived/clips/canonical.npz` | Stage 4 (feature engineering) and Stage 5 (skeleton model) |
| `derived/meta/canon_quality.csv` | quality reference |

**Confirm before moving on:**
1. Mirror-invariance test **PASSED** (cell 4)
2. Usable strokes **> 95%**
3. Every fold still has all 4 classes after dropping unusable strokes

Next: **`04_baseline.ipynb`** — ~45 hand-engineered scalars into LightGBM, target 62–70% macro-F1, with SHAP to show which kinematics actually carry the signal.
